# hs-classifier — Full Workflow

This notebook walks through the complete pipeline:

1. **Setup** — configure, build index, load classifier, smoke test
2. **Create eval sample** — take a representative slice of your data
3. **Label** — add ground truth HS codes (simulated here)
4. **Classify + evaluate** — classify the labeled sample and compute metrics
5. **Tune** — compare different model/parameter configurations
6. **Classify full dataset** — run the best config on all your data

## 1. Setup

In [1]:
import logging
import os
import warnings

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s")
for name in ("httpx", "faiss.loader", "sentence_transformers", "numba"):
    logging.getLogger(name).setLevel(logging.WARNING)

import pandas as pd
from sentence_transformers import SentenceTransformer

from hs_classifier import classify_row, init_classifier, init_index

In [2]:
# one-time: build the FAISS index from Atlas DB (skips if already exists)
init_index()

# load the classifier (FAISS index + S-BERT model)
classifier = init_classifier()

hs_classifier.init_lookup_index | HS chapters already exist at data/intermediate/hs2_chapters.parquet, skipping
hs_classifier.init_lookup_index | Index already exists at data/intermediate/hs12_4_index.parquet, skipping (use force=True to rebuild)
hs_classifier | Index initialization complete
hs_classifier | Loading HS chapters and FAISS index...
hs_classifier.retrieval | Index loaded: 1226 codes, 768d embeddings


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

hs_classifier | Classifier ready


## 2. Create an eval sample

Before classifying your full dataset, take a small representative sample using semantic clustering. The splitter only needs the text column — everything else passes through.

In [3]:
from hs_classifier.splitter import prepare_eval_sample

df = pd.read_csv("data/raw/ecuador_sample.csv")
print(f"Full dataset: {len(df)} rows")

embed_model = SentenceTransformer(
    os.environ["EMBEDDING_MODEL"], local_files_only=True
)
sample = prepare_eval_sample(
    df, text_col="product_description",
    model=embed_model, sample_frac=0.05,
)

print(f"\nEval sample: {len(sample)} rows ({len(sample)/len(df):.1%})")
sample.groupby("cluster").size().rename("n").sort_index().to_frame()

Full dataset: 600 rows


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

hs_classifier.splitter | Dropped 24 rows with null product_description
hs_classifier.splitter | Embedding 576 descriptions...


Batches:   0%|          | 0/18 [00:00<?, ?it/s]

hs_classifier.splitter | UMAP: 768d → 10d (n_neighbors=15)
hs_classifier.splitter | HDBSCAN: 17 clusters, 0 noise points
hs_classifier.splitter | Stratified sample: 30 rows from 576 (5.2%)



Eval sample: 30 rows (5.0%)


,n
cluster,
0,3
1,1
2,1
3,3
4,1
5,2
6,2
7,5
8,1


In [ ]:
# save the sample under the configured intermediate directory
sample.to_csv("data/intermediate/samples/ecuador_sample_eval_5pct.csv", index=False)
print("Saved to: data/intermediate/samples/ecuador_sample_eval_5pct.csv")

## 3. Label the sample

In practice, you'd now open the sample CSV and add an `hs_code` column with the correct HS4 code for each row — either manually, from existing labels, or with a labeling tool.

For this demo, we simulate labeled data with known ground truth:

In [5]:
# simulated labeled data — in practice this comes from your labeling step
labeled = pd.DataFrame({
    "product_description": [
        "frozen shrimp raw peeled",
        "fresh cut roses red",
        "fresh bananas green cavendish",
        "cacao beans raw fermented",
        "teak wood planks sawn",
        "gold unwrought bullion bars",
        "frozen lobster tails",
        "fresh carnations white",
        "organic bananas fair trade",
        "cocoa beans dried whole",
    ],
    "hs_code": ["0306", "0603", "0803", "1801", "4407", "7108", "0306", "0603", "0803", "1801"],
})
print(f"Labeled sample: {len(labeled)} rows")
labeled

Labeled sample: 10 rows


,product_description,hs_code
0,frozen shrimp raw peeled,0306
1,fresh cut roses red,0603
2,fresh bananas green cavendish,0803
3,cacao beans raw fermented,1801
4,teak wood planks sawn,4407
5,gold unwrought bullion bars,7108
6,frozen lobster tails,0306
7,fresh carnations white,0603
8,organic bananas fair trade,0803
9,cocoa beans dried whole,1801


## 4. Classify the labeled sample and evaluate

Run the classifier on each labeled row, collect predictions alongside ground truth, then compute metrics.

In [6]:
# classify each row and collect predictions
results = []
for row in labeled.to_dict(orient="records"):
    result = classify_row(row, classifier)
    results.append({
        "product_description": row["product_description"],
        "code_true": row["hs_code"],
        **{f"code_{i+1}": c for i, c in enumerate(result.codes)},
        "reason": result.reason,
    })

results_df = pd.DataFrame(results)
results_df

hs_classifier | Language: en | Query: frozen shrimp raw peeled
instructor.auto_client | Initializing anthropic provider with model claude-haiku-4-5-20251001
instructor.auto_client | Client initialized
hs_classifier | Search terms: ['frozen shrimp', 'crustaceans', 'aquatic invertebrates', 'peeled shrimp', 'fish and crustaceans preparations', 'raw seafood', 'frozen seafood products']
hs_classifier | Retrieved 24 candidate codes
instructor.auto_client | Initializing anthropic provider with model claude-haiku-4-5-20251001
instructor.auto_client | Client initialized
hs_classifier | Language: en | Query: fresh cut roses red
instructor.auto_client | Initializing anthropic provider with model claude-haiku-4-5-20251001
instructor.auto_client | Client initialized
hs_classifier | Search terms: ['cut flowers', 'fresh roses', 'ornamental foliage', 'live plants', 'floral products', 'horticultural crops', 'botanical specimens']
hs_classifier | Retrieved 27 candidate codes
instructor.auto_client | Ini

,product_description,code_true,code_1,code_2,reason
0,frozen shrimp raw peeled,0306,0306,1605,Frozen shrimp raw peeled is a crustacean produ...
1,fresh cut roses red,0603,0603,0602,Code 0603 directly covers cut flowers suitable...
2,fresh bananas green cavendish,0803,0803,0810,Code 0803 specifically covers bananas includin...
3,cacao beans raw fermented,1801,1801,1804,Code 1801 is the primary match for raw cacao b...
4,teak wood planks sawn,4407,4407,4403,Teak wood planks sawn are sawn wood products. ...
5,gold unwrought bullion bars,7108,7108,2616,Code 7108 directly covers gold unwrought or in...
6,frozen lobster tails,0306,0306,1605,Frozen lobster tails are crustaceans. Code 030...
7,fresh carnations white,0603,0603,0602,Fresh carnations are cut flowers suitable for ...
8,organic bananas fair trade,0803,0803,0810,Code 0803 directly covers bananas fresh or dri...
9,cocoa beans dried whole,1801,1801,1804,Code 1801 is the primary classification for co...


In [ ]:
from hs_classifier.evaluator import evaluation_report

report = evaluation_report(results_df, truth_col="code_true")

print(report["summary_text"])
print(report["top1_count"], report["topk_count"], report["chapter_count"])
print("\nCorrectness summary:")
print(report["correctness_summary"])

Top-1 matched on 10/10 rows, top-k matched on 10/10, and chapter matched on 10/10.
10/10 10/10 10/10
    metric  correct  incorrect  count
0     top1       10          0  10/10
1     topk       10          0  10/10
2  chapter       10          0  10/10


## 5. Tune and compare

Not happy with the results? Override model and retrieval parameters per call to compare configurations. No need to edit `.env` — just pass arguments to `classify_row()`.

In [ ]:
# compare two reranker models on the same labeled sample
configs = {
    "haiku": {"reranker_model": "anthropic/claude-haiku-4-5-20251001"},
    "gemini": {"reranker_model": "google/gemini-2.5-flash-lite"},
}

for name, params in configs.items():
    run_results = []
    for row in labeled.to_dict(orient="records"):
        result = classify_row(row, classifier, **params)
        run_results.append({
            "code_true": row["hs_code"],
            **{f"code_{i+1}": c for i, c in enumerate(result.codes)},
        })

    run_df = pd.DataFrame(run_results)
    report = evaluation_report(run_df, truth_col="code_true")
    top1 = report["top1_accuracy"]
    topk = report["topk_accuracy"]
    chap = report["chapter_accuracy"]
    print(f"[{name}] Top-1: {top1:.1%}  Top-K: {topk:.1%}  Chapter: {chap:.1%}")

In [ ]:
# or tune retrieval parameters
for top_k in [25, 50, 75]:
    run_results = []
    for row in labeled.to_dict(orient="records"):
        result = classify_row(row, classifier, top_k_total=top_k)
        run_results.append({
            "code_true": row["hs_code"],
            **{f"code_{i+1}": c for i, c in enumerate(result.codes)},
        })

    run_df = pd.DataFrame(run_results)
    report = evaluation_report(run_df, truth_col="code_true")
    top1 = report["top1_accuracy"]
    topk = report["topk_accuracy"]
    print(f"[top_k={top_k}] Top-1: {top1:.1%}  Top-K: {topk:.1%}")

## 6. Classify your full dataset

Once you've picked the best configuration, run it on your full dataset.

In [ ]:
# example: pick the best config from step 5, then classify everything
best_config = {"reranker_model": "google/gemini-2.5-flash-lite"}

df = pd.read_csv("data/raw/ecuador_sample.csv")

all_results = []
for row in df.to_dict(orient="records"):
    result = classify_row(row, classifier, **best_config)
    all_results.append({
        **row,
        **{f"hs_{i+1}": c for i, c in enumerate(result.codes)},
        "reason": result.reason,
    })

classified = pd.DataFrame(all_results)
output_path = "data/raw/ecuador_sample_classified.csv"
classified.to_csv(output_path, index=False)
print(f"Classified {len(classified)} rows")
print(f"Saved to: {output_path}")
classified.head()